# AMISGC — Emergent Intelligence from a Metabolically Constrained Predictive Neural Field

This notebook runs the full AMISGC research codebase on Kaggle. It mirrors the same engine that powers the web UI and the CLI.

**Phases included:** CORE-1, CORE-2, CORE-2.5, CORE-3.5, CORE-4, CORE-4.5, CORE-5/6/6.5, PHASE 7 → PHASE 12, embodiment battery, ablations, and the ARC mock benchmark.

**Scales supported:** 81, 810, and 81000 neurons.

Kaggle has Node.js available out of the box; this notebook installs `pnpm` once, sets up the workspace, then drives the same TypeScript engine via shell calls.

## 1. Environment setup

Install Node.js + pnpm and clone/upload the repository.

In [ ]:
import os, subprocess, shutil, json, sys

# Confirm Node.js is available (Kaggle ships with Node 18+)
subprocess.run(['node', '--version'], check=True)

# Install pnpm globally (needs network access enabled in Kaggle settings)
if not shutil.which('pnpm'):
    subprocess.run(['npm', 'install', '-g', 'pnpm@9'], check=True)
subprocess.run(['pnpm', '--version'], check=True)

In [ ]:
# Point this to where the AMISGC repo lives on Kaggle.
# Option A: upload the repo as a Kaggle dataset and copy it into /kaggle/working
# Option B: clone from your fork via git
REPO_DIR = '/kaggle/working/amisgc'
if not os.path.isdir(REPO_DIR):
    # Replace this URL with your fork.
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/YOUR_USER/amisgc.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pnpm', 'install', '--frozen-lockfile'], check=True)

## 2. List the experiment battery

All experiments are defined in `lib/amisgc-core/src/experiments.ts` and exposed via the CLI.

In [ ]:
!pnpm --filter @workspace/scripts amisgc list

## 3. Run a single experiment

Pick any experiment id from the battery above. Set the scale to 81, 810, or 81000.

In [ ]:
!pnpm --filter @workspace/scripts amisgc run C1.1 --scale=81

In [ ]:
!pnpm --filter @workspace/scripts amisgc run P8.1 --scale=810 --ticks=2000

## 4. ARC mock benchmark

The ARC benchmark trains the network on a small set of grid-style transformations (reverse, invert, shift, rotate, swap-pairs) and evaluates it on held-out probes.

In [ ]:
!pnpm --filter @workspace/scripts amisgc arc --scale=81 --tasks=8 --train=600 --test=3

## 5. Run the entire battery (long)

Sweeps every experiment at the given scale and reports a confirmed/rejected tally.

In [ ]:
!pnpm --filter @workspace/scripts amisgc all --scale=81

## 6. (Optional) Spin up the web UI on Kaggle

Start the API server and the React UI in the background. Use Kaggle's port-forwarding extensions (or `ngrok`) to view the UI in a browser.

In [ ]:
import subprocess, time
api = subprocess.Popen(['pnpm', '--filter', '@workspace/api-server', 'dev'])
ui = subprocess.Popen(['pnpm', '--filter', '@workspace/amisgc', 'dev'])
time.sleep(8)
print('api pid', api.pid, 'ui pid', ui.pid)
print('Open http://localhost:5173 (or whichever PORT was assigned).')